# <center> Encoding Data
## <center> SYSE 549: Secure Vehicle and Industrial Networking
## <center><img src="https://www.engr.colostate.edu/~jdaily/Systems-EN-CSU-1-C357.svg" width="400" />
### <center> Instructor: Dr. Jeremy Daily

## Lesson Outcomes

After completing this exercise, students should be able to:

1. Realize encoded data is **not** encrypted.
2. Carry out data encoding for integers of different lengths.
3. Present data as text strings or numbers based on the desired encoding.
4. Develop Python programming skills to work with different types of encoding.

### Why this matters for vehicle and industrial networks

A CAN frame carries at most 8 bytes. Those 8 bytes are just 64 electrical decisions on a
twisted pair. They have no inherent meaning. Meaning comes from a *convention* that both the
transmitter and receiver agree on, written down in a standard such as SAE J1939, NMEA 2000, or
ISO 14229 (UDS).

That is the whole game for a network analyst: you capture bytes, then you decide what they mean.
If you guess the convention wrong, you get a plausible-looking number that is completely false.
This notebook is about developing the discipline to get the convention right, and about
recognizing that any attacker with the same standard on their desk can read the traffic just as
easily as you can.

### Overview

1. Define a string of bytes
2. Explore integer encoding
3. Learn the `struct` library
4. Understand how to encode data as text with different codecs
5. Represent binary using only text with Base64 encoding
6. Look at some crude ciphers and see why they fail

### How to work through this notebook

Run the cells **in order** — later cells depend on variables defined earlier.

Before you press `Shift+Enter` on a cell, take five seconds and *predict the output*. Say the
prediction out loud or write it in the margin. Then run the cell. The moments where your
prediction is wrong are the moments you are actually learning something; the rest is typing.

---
## 1. One Number, Many Faces

Everything in this section is the *same 64 bits*. Only the presentation changes.

Python lets you write an integer literal directly in binary using the `0b` prefix. Underscores
are legal as visual separators if you want them (`0b0110_0001`), and they are ignored by the
interpreter.

In [1]:
# Given a series of bits.
# This could represent the 8-byte data field of a CAN frame.
# The 0b prefix tells Python to read the following digits as binary.
a = 0b0110000101110100011101000110000101100011011010110000110100001010

In [2]:
# What is a?
# Python has no "bit string" type, so the literal above became an integer.
print(a)
type(a)

7022365680606055690


int

That base-10 number is technically correct and completely useless to a human trying to read a
network capture. Hexadecimal is the standard presentation for raw data because **each hex digit
maps to exactly 4 bits**, so the grouping never lies to you.

The format specifier `{:016X}` means: format as uppercase heXadecimal, zero-padded to 16
characters. Sixteen hex digits x 4 bits per digit = 64 bits.

In [3]:
# Display the integer as hex characters.
# 016X -> zero-pad to 16 digits, uppercase hex. 16 digits x 4 bits = 64 bits.
print("{:016X}".format(a))

61747461636B0D0A


In [4]:
# Display the integer as binary.
# 064b -> zero-pad to 64 digits, binary. Compare this to the literal we typed above.
print("{:064b}".format(a))

0110000101110100011101000110000101100011011010110000110100001010


The `.format()` method works everywhere, but f-strings (Python 3.6+) are usually easier to read
because the variable name sits inside the string where it is used. The format specifier after the
colon is identical in both styles.

In [5]:
# Use an f-string (f prefix). The specifier after the colon is the same as in .format().
f"{a:064b}"

'0110000101110100011101000110000101100011011010110000110100001010'

---
## 2. From Integer to `bytes`

Python 3 has a dedicated `bytes` type. Almost everything you receive from a socket, a serial
port, a CAN interface, or a file read in binary mode arrives as `bytes`.

`int.to_bytes(length, byteorder)` converts an integer into a fixed-width byte string. Both
arguments matter:

- **length** — how many bytes to use. Too few raises `OverflowError`; too many pads with zeros.
- **byteorder** — `'big'` puts the most significant byte first (the way humans write numbers);
  `'little'` puts the least significant byte first.

Watch what Python does when it *displays* the result.

In [6]:
# Convert the integer into 8 bytes, most significant byte first ('big').
b = a.to_bytes(8, 'big')
print(b)
type(b)

b'attack\r\n'


bytes

Python printed `b'attack\r\n'` rather than a row of hex.

This is a display convenience, not a conversion. The `repr()` of a `bytes` object shows any byte
that happens to fall in the printable ASCII range as its ASCII character, and escapes the rest.
The object is still eight raw bytes. It is not a string, and Python will refuse to concatenate it
with one.

This is worth internalizing now, because it will bite you later: **the way Python displays bytes
is a guess about what you meant.** Our 8 bytes could equally well be two 32-bit numbers representing distance values.

In [7]:
# Confirm the length is what we asked for.
len(b)

8

In [8]:
# You can iterate through a bytes object.
# Note that indexing/iterating a bytes object yields *integers*, not 1-byte slices.
for i in b:
    print(i, end=' ')

97 116 116 97 99 107 13 10 

In [9]:
# Same loop, but format each integer as 2 hex digits.
for i in b:
    print("{:02X}".format(i), end=' ')

61 74 74 61 63 6B 0D 0A 

### 2.1 Making bytes readable

The `.hex()` method is the quickest way to get a hex string, but it runs the digits together.
For anything you intend to read with your eyes, separate the bytes.

The idiom below is a **list comprehension** — a compact way to build a list by transforming every
element of an iterable. Read `[f(i) for i in b]` as "the list of `f(i)`, for each `i` in `b`."
It is worth learning; you will use it constantly when massaging captured data.

In [10]:
# Default rendering: one long run of hex digits, no separators.
b.hex()

'61747461636b0d0a'

In [11]:
# A nicer display of hex from raw bytes.
# The list comprehension formats each byte; " ".join() glues them with spaces.
" ".join(["{:02X}".format(i) for i in b])

'61 74 74 61 63 6B 0D 0A'

In [12]:
# The same idea, but showing all 64 bits grouped into bytes.
" ".join(["{:08b}".format(i) for i in b])

'01100001 01110100 01110100 01100001 01100011 01101011 00001101 00001010'

### 2.2 The same bytes, a different story

Suppose a protocol dissector had been told these bytes were network addresses. Here is what the
*exact same data* would look like on screen. Nothing changed but the assumed convention.

In [13]:
# Pretend the last 4 bytes are an IPv4 address.
# Dotted-quad notation is just each byte printed as a decimal number.
print("IP: " + ".".join(["{:}".format(i) for i in b[-4:]]))

IP: 99.107.13.10


In [14]:
# Pretend the first 4 bytes are a different address.
# A packet analyzer such as Wireshark would show the raw field like this.
" ".join(["{:02X}".format(i) for i in b[:4]])

'61 74 74 61'

### 2.3 Round trip

Encoding should be reversible. If it is not, you have lost information somewhere and you should
find out where.

In [15]:
# Convert the hex string back to an integer (base 16).
int(b.hex(), 16)

7022365680606055690

In [16]:
# Compare to the original integer. These should match exactly.
a

7022365680606055690

---
## 3. Decoding Options: the `struct` Module

There are many ways to decode raw bytes. The `struct` module is the standard tool: it converts
between Python values and C-style packed binary data, which is exactly what embedded controllers
put on the wire.

Reference: https://docs.python.org/3/library/struct.html

### Format characters

| Code | C type | Python type | Standard size |
| :--- | :--- | :--- | :--- |
| `x` | pad byte | *no value* | 1 |
| `c` | `char` | `bytes` of length 1 | 1 |
| `b` | `signed char` | `int` | 1 |
| `B` | `unsigned char` | `int` | 1 |
| `h` | `short` | `int` | 2 |
| `H` | `unsigned short` | `int` | 2 |
| `i` / `l` | `int` / `long` | `int` | 4 |
| `I` / `L` | `unsigned int` / `unsigned long` | `int` | 4 |
| `q` | `long long` | `int` | 8 |
| `Q` | `unsigned long long` | `int` | 8 |
| `f` | `float` | `float` | 4 |
| `d` | `double` | `float` | 8 |
| `s` | `char[]` | `bytes` | as counted |

**Lowercase is signed, uppercase is unsigned.** A leading count repeats the code, so `4H` and
`HHHH` mean the same thing.

### Byte-order prefixes

| Prefix | Byte order | Size and alignment |
| :--- | :--- | :--- |
| `@` | native | native, **with padding** (this is the default!) |
| `=` | native | standard, no padding |
| `<` | little-endian (Intel) | standard, no padding |
| `>` | big-endian (Motorola, "network order") | standard, no padding |
| `!` | network (same as `>`) | standard, no padding |

> **Always write the prefix.** With no prefix you get `@`, which follows whatever your CPU
> happens to do. An x86 laptop is little-endian, so your code will
> appear to work. Move it to a big-endian target, or add a field that triggers alignment padding,
> and it will silently produce garbage. Being explicit costs one character.

In [17]:
# The struct module is part of the standard library; nothing to install.
import struct

In [18]:
# Recall our 8 bytes.
b

b'attack\r\n'

In [19]:
# Interpretation 1: eight single-byte UNSIGNED integers (range 0..255).
struct.unpack("BBBBBBBB", b)

(97, 116, 116, 97, 99, 107, 13, 10)

In [20]:
# Interpretation 2: eight single-byte SIGNED integers (range -128..127).
# 8b is shorthand for bbbbbbbb.
struct.unpack("8b", b)

(97, 116, 116, 97, 99, 107, 13, 10)

Those two results are identical, which is not a coincidence: every byte in this particular
message is below 128, so the sign bit is clear and the two interpretations agree.

Pick a byte with the high bit set and the illusion collapses.

In [21]:
# A single byte with the most significant bit set.
# The \x escape means "the next two characters are hex digits."
c = b'\xDA'
print(c)

b'\xda'


In [22]:
# Unsigned interpretation: 1101 1010 = 218.
struct.unpack('B', c)

(218,)

In [23]:
# Signed interpretation: the same bits in two's complement = -38.
# Note 218 - 256 = -38.
struct.unpack('b', c)

(-38,)

Same electrical signal. Two answers, 256 apart. Only the data definition tells you which is
correct — and a sensor reading of `-38` instead of `218` is exactly the kind of error that
propagates quietly into a report.

Notice also that `unpack` returned `(218,)` with a trailing comma. `unpack` **always** returns a
tuple, even for a single value, because a format string can describe any number of fields.

In [24]:
# The return value is always a tuple.
type(struct.unpack('b', c))

tuple

In [25]:
# To get a plain integer, index into the tuple.
struct.unpack('b', c)[0]

-38

In [26]:
# Confirm we now have an int rather than a tuple.
type(struct.unpack('b', c)[0])

int

### 3.1 Endianness: 16-bit integers

Now we take the same 8 bytes two at a time. Each pair becomes one 16-bit value, and the order in
which the two bytes are combined changes the answer.

In [27]:
# Four 16-bit unsigned integers, using the NATIVE byte order (no prefix).
# On an x86 machine this is little-endian, but do not rely on that.
struct.unpack("4H", b)

(29793, 24948, 27491, 2573)

In [30]:
# Explicitly little-endian (Intel / J1939 data order): least significant byte first.
struct.unpack("<HHHH", b)

(29793, 24948, 27491, 2573)

In [31]:
# Explicitly big-endian (Motorola / human order): most significant byte first.
struct.unpack(">HHHH", b)

(24948, 29793, 25451, 3338)

Compare the three results. The native-order run matched little-endian *on this machine*.
That agreement is a property of your laptop, not of your code.

Now run the conversion in the other direction with `pack`, which is the inverse of `unpack`.

In [32]:
# Big endian (the way humans write numbers: most significant digit on the left).
(24948).to_bytes(2, 'big')

b'at'

In [33]:
# The same operation using struct. Motorola format.
struct.pack(">H", 24948)

b'at'

In [34]:
# Little endian: reversed byte order.
(24948).to_bytes(2, 'little')

b'ta'

In [35]:
# Little endian using struct (Intel format).
# Note 24948 == 0x6174, so the bytes are 0x61 and 0x74.
d = struct.pack("<H", 0x6174)
print(d)

b'ta'


In [36]:
# Confirm the reversed byte order: 74 61 instead of 61 74.
" ".join(["{:02X}".format(i) for i in d])

'74 61'

### 3.2 Signed 16-bit values

`h` is the signed counterpart of `H`. Combine signedness with endianness and there are four
plausible readings of any 2-byte field. This is why a J1939 or NMEA 2000 data dictionary
specifies both.

In [37]:
# Signed 2-byte integer, big endian. 0xDA55 has the sign bit set.
struct.unpack(">h", b'\xda\x55')

(-9643,)

In [38]:
# Signed 2-byte integer, little endian. Same bytes, different value entirely.
struct.unpack("<h", b'\xda\x55')

(21978,)

In [39]:
# The little-endian reading is the positive number 0x55DA.
print(0x55da)

21978


---
## 4. Bytes as Characters

Text is just another encoding. A codec is a lookup table that assigns a meaning to each byte
value (or each byte *sequence*, for the multi-byte codecs).

In [40]:
# Interpretation 3: eight single characters.
# Unlike 'B', the 'c' code returns 1-byte bytes objects rather than integers.
e = struct.unpack("8c", b)
print(e)

(b'a', b't', b't', b'a', b'c', b'k', b'\r', b'\n')


In [41]:
# Glue the tuple of 1-byte objects back into a single bytes object.
# b'' is an empty bytes object used as the separator.
b''.join(e)

b'attack\r\n'

In [42]:
# The same result can be built directly with pack.
f = struct.pack("cccccccc", b'a', b't', b't', b'a', b'c', b'k', b'\r', b'\n')
print(f)

b'attack\r\n'


In [43]:
# Using 's' takes a whole byte string at once, so no join is needed.
# "8s" means "8 bytes treated as one field."
f = struct.pack("8s", b'attack\r\n')
print(f)

b'attack\r\n'


In [44]:
# f is still a bytes object, not a str.
type(f)

bytes

### 4.1 `str()` is not decoding

A common beginner mistake is calling `str()` on a `bytes` object. Python obliges, but it gives
you the *representation* — including the `b` prefix and the quote marks — as literal characters.
That is almost never what you want.

To convert bytes to text you must name a codec with `.decode()`.

In [45]:
# Convert to a string the wrong way: this stringifies the repr, prefix and all.
str(f)

"b'attack\\r\\n'"

In [46]:
# The right way: decode with an explicit codec. We know this is ASCII text.
f.decode('ascii')

'attack\r\n'

In [47]:
# Printing renders the escapes: \r is a carriage return, \n a line feed.
# Together they are the CRLF line ending used by many protocols.
print(f.decode('ascii'))

attack



### 4.2 Choosing a codec

ASCII defines only 128 values (0x00-0x7F). Every byte with the high bit set is undefined, and
different codecs disagree about what those bytes mean:

- **UTF-8** — the modern default. Backward compatible with ASCII for 0x00-0x7F; uses multi-byte
  sequences above that. Not every byte sequence is valid UTF-8.
- **Latin-1** (ISO 8859-1) — a single-byte codec that maps all 256 values to *something*. It can
  never fail, which makes it useful for round-tripping arbitrary bytes through a text pipeline.
- **ASCII** — 7-bit only; raises on anything above 0x7F.

In [48]:
# UTF-8 is the modern default, and matches ASCII for these byte values.
f.decode('utf-8')

'attack\r\n'

In [49]:
# Latin-1 is a different character set that also agrees here.
f.decode('latin-1')

'attack\r\n'

In [50]:
# Now add two bytes above 0x7F. Latin-1 maps every byte to some character.
b'attack\xc4\xfa'.decode('latin-1')

'attackÄú'

In [51]:
# The same bytes as ASCII: 0xC4 is outside the 7-bit range, so this raises.
# EXPECT AN EXCEPTION HERE. Read the traceback; it names the offending byte.
b'attack\xc4\xfa'.decode('ascii')

UnicodeDecodeError: 'ascii' codec can't decode byte 0xc4 in position 6: ordinal not in range(128)

### 4.3 Error handlers

The second argument to `.decode()` chooses what happens when a byte cannot be decoded:

| Handler | Behavior |
| :--- | :--- |
| `'strict'` | raise `UnicodeDecodeError` (the default) |
| `'ignore'` | silently drop the offending bytes |
| `'replace'` | substitute U+FFFD, the replacement character |

`'ignore'` is convenient and dangerous: it destroys data without telling you. Use it when you are
eyeballing a capture, never in a parser whose output someone will rely on.

In [52]:
# 'ignore' drops the bytes it cannot handle. Note the data loss is silent.
b'attack\xc4\xfa'.decode('ascii', 'ignore')

'attack'

In [53]:
# UTF-8 encodes characters outside ASCII as multi-byte sequences.
truck = '\U0001F69B'.encode('utf-8')
truck

b'\xf0\x9f\x9a\x9b'

In [54]:
# How many bytes did that one character take?
len(truck)

4

In [55]:
# Decoding reverses it exactly.
truck.decode('utf-8')

'🚛'

In [56]:
# 0xC4 0xFA is not a legal UTF-8 sequence, so strict decoding raises.
# EXPECT AN EXCEPTION HERE.
b'attack\xc4\xfa'.decode('utf-8')

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc4 in position 6: invalid continuation byte

In [57]:
# 'replace' keeps the pipeline running and marks the damage visibly.
b'attack\xc4\xfa'.decode('utf-8', 'replace')

'attack��'

In [58]:
# By contrast, 0xC4 0x8A IS a valid 2-byte UTF-8 sequence, so strict mode succeeds.
b'attack\xc4\x8a'.decode('utf-8', 'strict')

'attackĊ'

---
## 5. Long Integers (32-bit)

Four bytes at a time. This is the width used for odometers, engine hours, and most cumulative
counters in J1939, because 16 bits would roll over far too quickly.

In [59]:
# Two 32-bit unsigned integers, big endian.
struct.unpack('>LL', b)

(1635021921, 1667960074)

In [60]:
# The largest value a 32-bit unsigned integer can hold, plus one.
2**32

4294967296

In [61]:
# The same bytes, little endian.
struct.unpack('<LL', b)

(1635021921, 168651619)

The *first* value came out identical in both byte orders. That is not a bug: the first four bytes
spell `atta`, whose byte sequence `61 74 74 61` is a palindrome. Reversing it changes nothing.

The second four bytes are not palindromic, and there the two readings diverge completely.

This is a useful reminder when you are reverse-engineering an unfamiliar protocol: a field that
"works" in both byte orders has told you nothing about the endianness. You need a field whose
value you can independently verify.

In [62]:
# Isolate the last four bytes and read them little endian.
struct.unpack('<L', b'ck\r\n')

(168651619,)

In [63]:
# The same four bytes read big endian.
struct.unpack('>L', b'ck\r\n')

(1667960074,)

In [64]:
# Pack the little-endian value back to bytes to confirm the round trip.
struct.pack("<L", 168651619)

b'ck\r\n'

In [65]:
# That decimal number written in hex. Compare it to the bytes above.
0x0A0D6B63

168651619

In [66]:
# Packing the same integer big endian reverses the byte order.
struct.pack(">L", 0x0A0D6B63)

b'\n\rkc'

In [67]:
# Signed 32-bit integers. Both values here are below 2**31, so they match the unsigned reading.
struct.unpack('>ll', b)

(1635021921, 1667960074)

---
## 6. Practical Example: Decoding Vehicle Miles

SAE J1939 defines PGN 65248, *Vehicle Distance*. It carries two 32-bit integers in the 8-byte
data field:

| Bytes | SPN | Parameter | Resolution |
| :--- | :--- | :--- | :--- |
| 1-4 | 244 | Trip Distance | 0.125 km/bit |
| 5-8 | 245 | Total Vehicle Distance (odometer) | 0.125 km/bit |

J1939 numbers bytes starting at 1; Python indexes from 0. Keep that offset straight or you will
decode the trip meter and call it the odometer.

J1939 data fields are **little endian**. The 29-bit CAN identifier, on the other hand, is read
most significant bit first. Mixed conventions in one protocol are normal, and are a reliable
source of bugs.

In [68]:
# First, what does PGN 65248 look like in hex? Look for it inside the CAN ID below.
"{:X}".format(65248)

'FEE0'

In [69]:
# A line from a CAN log captured on a truck.
# Fields: (timestamp) channel  CAN-ID  [length] data bytes...
log_text = "(012.102753)  can1  18FEE000   [8] 73 49 03 00 BC E0 33 00"

In [70]:
# split() with no argument splits on any run of whitespace, so the double
# spaces in the log line do not produce empty entries.
entries = log_text.split()
entries

['(012.102753)',
 'can1',
 '18FEE000',
 '[8]',
 '73',
 '49',
 '03',
 '00',
 'BC',
 'E0',
 '33',
 '00']

The CAN ID is `18FEE000`. Breaking it apart: priority `0x18` (which carries priority 6 in its
upper bits), PDU Format `0xFE`, PDU Specific `0xE0`, and source address `0x00` — the engine
controller. `FEE0` is our PGN 65248. Extracting those fields properly is the subject of the
next notebook.

Three ways to turn the trailing hex text into bytes follow. All three produce identical results;
they differ in clarity and in how gracefully they handle a malformed line.

In [71]:
# Method 1: join the last 8 fields into one hex string, then convert in one call.
# This is the clearest of the three.
data_bytes = bytes.fromhex(''.join(entries[-8:]))
data_bytes

b'sI\x03\x00\xbc\xe03\x00'

In [72]:
# Method 2: convert each hex field to an int, then build bytes from the list.
data_bytes = bytes([int(i, 16) for i in entries[-8:]])
data_bytes

b'sI\x03\x00\xbc\xe03\x00'

In [73]:
# Method 3: build up the bytes one at a time (NOT preferred).
# Repeated concatenation creates a new object each pass; it is slower and noisier.
data_bytes = b''
for i in entries[4:12]:
    data_bytes += int(i, 16).to_bytes(1, 'big')
data_bytes

b'sI\x03\x00\xbc\xe03\x00'

In [74]:
# J1939 data is little endian (Intel format).
# Two unsigned 32-bit values: (SPN 244 trip, SPN 245 total).
pgn_values = struct.unpack('<LL', data_bytes)
pgn_values

(215411, 3399868)

In [75]:
# Apply the SPN 245 scaling: 0.125 km per bit, then convert km to miles.
# 1 mile == 1.609344 km exactly, by international agreement.
SPN245 = 0.125 * pgn_values[1] / 1.609344
print("The Total Vehicle Distance is {:0,.1f} miles.".format(SPN245))

The Total Vehicle Distance is 264,072.5 miles.


In [76]:
# Same treatment for SPN 244, the trip distance.
SPN244 = 0.125 * pgn_values[0] / 1.609344
print("The Trip Distance is {:0,.1f} miles.".format(SPN244))

The Trip Distance is 16,731.3 miles.


### 6.1 Doing it the long way

`struct.unpack` is not magic. A little-endian integer is a base-256 place-value number: the first
byte is the ones place, the next is the 256s place, and so on.

Working through it by hand once makes the byte-order question concrete, and it is exactly the
arithmetic you would write in C on a controller with no `struct` module available.

In [77]:
# The long way: multiply each byte by its place value.
# Bytes 5-8 in J1939 numbering are indices 4-7 in Python.
value = 0
value += data_bytes[4]                # 256**0, the ones place
value += data_bytes[5] * 256          # 256**1
value += data_bytes[6] * 256 * 256    # 256**2
value += data_bytes[7] * 256 * 256 * 256   # 256**3
value

3399868

In [78]:
# Confirm the hand calculation matches what struct.unpack produced.
value == pgn_values[1]

True

---
## 7. 64-bit Numbers and Floating Point

Eight bytes read as a single field. `Q` and `q` are the unsigned and signed 64-bit integers;
`d` is an IEEE 754 double-precision float; `f` is a 4-byte single-precision float.

In [79]:
# Pack the original 64-bit integer back into bytes, big endian.
# This should reproduce b exactly.
struct.pack('>Q', a)

b'attack\r\n'

In [80]:
# And convert back to a 64-bit unsigned integer.
struct.unpack('>Q', b)

(7022365680606055690,)

In [81]:
# Byte order still matters, and matters a great deal at this width.
struct.unpack('<Q', b)

(724353189657474145,)

In [82]:
# Signed 64-bit. Our value is below 2**63, so it is unchanged.
struct.unpack('>q', b)

(7022365680606055690,)

In [83]:
# Setting the most significant bit of the first byte makes the value negative.
# 0x64 ('d') becomes 0xD0, so the sign bit is now set.
neg_num = struct.unpack('>q', b'\xd0ttack\r\n')
neg_num

(-3425985454893495030,)

### 7.1 The same bytes as floating point

IEEE 754 packs a sign, an exponent, and a mantissa into a fixed-width field. Feed it bytes that
were never meant to be a float and it will happily return a number — usually an absurd one.

An implausible magnitude (10^161, or 10^-260) is a strong hint that you have applied the wrong
interpretation. Absurdity is a diagnostic. Learn to notice it.

In [84]:
# Read the 8 bytes as two 4-byte floats, big endian.
struct.unpack('>ff', b)

(2.8183697112289195e+20, 4.335924420794005e+21)

In [85]:
# The same bytes as two little-endian floats.
# The first value is unchanged because 'atta' is a byte palindrome; the second is not.
struct.unpack('<ff', b)

(2.8183697112289195e+20, 6.809100250964041e-33)

In [86]:
# All 8 bytes as one double-precision float, big endian.
struct.unpack('>d', b)[0]

2.8757353661668934e+161

In [87]:
# All 8 bytes as one double, little endian. Note the enormous difference.
struct.unpack('<d', b)[0]

2.989708374342575e-260

Roughly 2.9 x 10^161 one way and 3.0 x 10^-260 the other. These are not related by any arithmetic
operation on the value — not a reciprocal, not a sign flip, not a scale factor. Reversing bytes is
an operation on the *representation*, and once the exponent field has been rearranged the
resulting number bears no meaningful relationship to the original.

Most CAN networks do not use floating point or doubles on the network. Instead, they use Scale and Offset to get decimal numbers.

---
## 8. Sending Bytes as Text Only: Base64

Many transports are hostile to arbitrary bytes. Email, JSON, URLs, XML, and plenty of logging
systems will mangle control characters, or treat 0x00 as an end-of-string, or reflow whitespace.

Base64 solves this by regrouping the data into 6-bit chunks and mapping each chunk to one of 64
safe ASCII characters (`A-Z`, `a-z`, `0-9`, `+`, `/`), with `=` used as padding. Three bytes in
become four characters out.

Reference: https://docs.python.org/3/library/base64.html

This is how cryptographic keys and certificates get carried inside text files.

In [88]:
# base64 is in the standard library.
import base64

In [89]:
# Encode our 8 bytes. The result is itself a bytes object, containing only safe ASCII.
g = base64.b64encode(b)
print(g)

b'YXR0YWNrDQo='


In [90]:
# Decoding restores the original bytes exactly. No key, no secret, no loss.
base64.b64decode(g)

b'attack\r\n'

In [91]:
# Because the output is all printable ASCII, it converts to a str safely.
g.decode('utf-8')

'YXR0YWNrDQo='

In [92]:
# Length of the encoded form.
len(g)

12

In [93]:
# Length of the original.
len(b)

8

### 8.1 The cost of Base64

Base64 expands data by a factor of **4/3** (four output characters per three input bytes), plus
up to two padding characters.

Here 8 bytes became 12 characters, a ratio of exactly 1.5 — that is 4/3 rounded up to a multiple
of 4 plus one `=` of padding. For larger inputs the padding becomes negligible and the ratio
converges on 1.333.

Compare that with plain hex, which needs two characters per byte: a fixed factor of 2. Base64 is
the more efficient of the two text-safe encodings, which is why it won.

In [94]:
# Recall the raw bytes.
b

b'attack\r\n'

In [95]:
# What about simply converting to hex characters instead?
h = b.hex()
h

'61747461636b0d0a'

In [96]:
# Hex doubles the length: 2 characters per byte, always.
len(h)

16

In [97]:
# Same content again, as we formatted it back in section 1.
print("{:016X}".format(a))

61747461636B0D0A


In [98]:
# Anyone can decode this. There is no key involved.
base64.b64decode('YXR0YWNrDQo=')

b'attack\r\n'

### 8.2 Measuring the expansion properly

Eight bytes is too small a sample to see the true ratio, because padding dominates. Let's encode
all 256 possible byte values and measure again.

In [99]:
# Generate a list containing every possible byte value, 0x00 through 0xFF.
char_list = [struct.pack("B", i) for i in range(256)]
# Print only the first 16 so the output stays readable.
print(char_list[:16])

[b'\x00', b'\x01', b'\x02', b'\x03', b'\x04', b'\x05', b'\x06', b'\x07', b'\x08', b'\t', b'\n', b'\x0b', b'\x0c', b'\r', b'\x0e', b'\x0f']


In [100]:
# Join them into one 256-byte object and encode it.
j = base64.b64encode(b''.join(char_list))
print(j)

b'AAECAwQFBgcICQoLDA0ODxAREhMUFRYXGBkaGxwdHh8gISIjJCUmJygpKissLS4vMDEyMzQ1Njc4OTo7PD0+P0BBQkNERUZHSElKS0xNTk9QUVJTVFVWV1hZWltcXV5fYGFiY2RlZmdoaWprbG1ub3BxcnN0dXZ3eHl6e3x9fn+AgYKDhIWGh4iJiouMjY6PkJGSk5SVlpeYmZqbnJ2en6ChoqOkpaanqKmqq6ytrq+wsbKztLW2t7i5uru8vb6/wMHCw8TFxsfIycrLzM3Oz9DR0tPU1dbX2Nna29zd3t/g4eLj5OXm5+jp6uvs7e7v8PHy8/T19vf4+fr7/P3+/w=='


In [101]:
# Length of the encoded form.
len(j)

344

In [102]:
# Length of the input.
len(char_list)

256

In [103]:
# The expansion ratio, now very close to 4/3 = 1.3333...
len(j) / len(char_list)

1.34375

> **Base64 encoded data is NOT encrypted.**
>
> No additional information is required to decode it. There is no key. Anyone with a browser can
> reverse it in one second.
>
> Encoding answers the question *"how do I represent this?"* Encryption answers *"who is allowed
> to read this?"* Confusing the two is one of the most common findings in security assessments of
> embedded and telematics systems. If you find credentials "protected" by Base64 in a device you
> are testing, they are stored in plaintext, and you should write it up that way.

---
## 9. Crude Ciphers

Now let's actually try to hide something, and watch it fail. The point of this section is not to
teach you ciphers you should use — it is to build the instinct that a scheme without a real key,
or with a tiny key, provides no protection at all.

### 9.1 Single-byte XOR

XOR has a property that makes it attractive for encryption: it is its own inverse. Applying the
same key twice returns the original data.

| A | B | A XOR B |
| :-: | :-: | :-: |
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

In [104]:
# Our plaintext. Recognizable English is what will give the cipher away.
plain_text = "Fourscore and seven years ago our fathers brought forth, on this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived, and so dedicated, can long endure."

In [105]:
# A one-byte key. That is the entire secret: 8 bits, 256 possibilities.
key = 5
struct.pack('B', key)

b'\x05'

In [106]:
# Encode the text to bytes before doing arithmetic on it.
plain_bytes = bytes(plain_text, 'utf-8')
print(plain_bytes)

b'Fourscore and seven years ago our fathers brought forth, on this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived, and so dedicated, can long endure.'


In [107]:
# XOR every byte with the key. The generator expression feeds bytes() one value at a time.
cipher_bytes = bytes(x ^ key for x in plain_bytes)
cipher_bytes

b'Cjpwvfjw`%dka%v`s`k%|`dwv%dbj%jpw%cdqm`wv%gwjpbmq%cjwqm)%jk%qmlv%fjkqlk`kq)%d%k`r%kdqljk)%fjkf`ls`a%lk%ilg`wq|)%dka%a`alfdq`a%qj%qm`%uwjujvlqljk%qmdq%dii%h`k%dw`%fw`dq`a%`tpdi+%Kjr%r`%dw`%`kbdb`a%lk%d%bw`dq%flsli%rdw)%q`vqlkb%rm`qm`w%qmdq%kdqljk)%jw%dk|%kdqljk%vj%fjkf`ls`a)%dka%vj%a`alfdq`a)%fdk%ijkb%`kapw`+'

In [108]:
# Decryption is the identical operation, because XOR is its own inverse.
bytes(x ^ key for x in cipher_bytes)

b'Fourscore and seven years ago our fathers brought forth, on this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived, and so dedicated, can long endure.'

### 9.2 Breaking it without the key

The ciphertext looks like noise, which feels reassuring. It should not.

Printable English text occupies a narrow band of byte values: the space character (0x20 = 32) is
the smallest, and `~` (0x7E = 126) is the largest. That constraint is enough to recognize a
correct decryption automatically.

With only 256 possible keys, we can simply try them all and keep any result that lands entirely
inside the printable range. This is a **known-plaintext-structure attack**, and it takes
microseconds.

In [109]:
# The smallest byte value in ordinary English text is the space character.
min(plain_bytes)

32

In [110]:
# Confirm that byte 32 is indeed a space.
struct.pack("B", min(plain_bytes))

b' '

In [111]:
# What if you don't know the key? Try all 256 of them.
# Keep any candidate that decodes entirely into the printable ASCII range.
for k in range(256):
    candidate_bytes = bytearray(x ^ k for x in cipher_bytes)
    if min(candidate_bytes) >= 32 and max(candidate_bytes) < 127:  # then ASCII
        print(k)
        print(candidate_bytes)
        print()

0
bytearray(b'Cjpwvfjw`%dka%v`s`k%|`dwv%dbj%jpw%cdqm`wv%gwjpbmq%cjwqm)%jk%qmlv%fjkqlk`kq)%d%k`r%kdqljk)%fjkf`ls`a%lk%ilg`wq|)%dka%a`alfdq`a%qj%qm`%uwjujvlqljk%qmdq%dii%h`k%dw`%fw`dq`a%`tpdi+%Kjr%r`%dw`%`kbdb`a%lk%d%bw`dq%flsli%rdw)%q`vqlkb%rm`qm`w%qmdq%kdqljk)%jw%dk|%kdqljk%vj%fjkf`ls`a)%dka%vj%a`alfdq`a)%fdk%ijkb%`kapw`+')

1
bytearray(b'Bkqvwgkva$ej`$waraj$}aevw$eck$kqv$beplavw$fvkqclp$bkvpl($kj$plmw$gkjpmjajp($e$jas$jepmkj($gkjgamra`$mj$hmfavp}($ej`$`a`mgepa`$pk$pla$tvktkwmpmkj$plep$ehh$iaj$eva$gvaepa`$auqeh*$Jks$sa$eva$ajceca`$mj$e$cvaep$gmrmh$sev($pawpmjc$slaplav$plep$jepmkj($kv$ej}$jepmkj$wk$gkjgamra`($ej`$wk$`a`mgepa`($gej$hkjc$aj`qva*')

2
bytearray(b"Ahrutdhub\'fic\'tbqbi\'~bfut\'f`h\'hru\'afsobut\'euhr`os\'ahuso+\'hi\'sont\'dhisnibis+\'f\'ibp\'ifsnhi+\'dhidbnqbc\'ni\'knebus~+\'fic\'cbcndfsbc\'sh\'sob\'wuhwhtnsnhi\'sofs\'fkk\'jbi\'fub\'dubfsbc\'bvrfk)\'Ihp\'pb\'fub\'bi`f`bc\'ni\'f\'`ubfs\'dnqnk\'pfu+\'sbtsni`\'pobsobu\'sofs\'ifsnhi+\'hu\'fi~\'ifsnhi\'th\'dhidbnqbc+\'fic\'th\'c

One candidate survived, and it is the plaintext. The key space was so small that "secret" never
meant anything.

Note what did the work here: not cleverness, but the fact that the *structure* of the plaintext
leaked through the cipher. Real ciphers are designed so that no property of the plaintext
survives. Every key should produce output that looks equally plausible.

### 9.3 The Caesar shift

An even older idea: add a constant to every value instead of XORing. It fails the same way, and
to a slightly more elegant attack.

In [112]:
# A Caesar shift cipher: add the key to every byte.
shifted_text = bytes((x + key) for x in plain_bytes)
shifted_text

b'Ktzwxhtwj%fsi%xj{js%~jfwx%flt%tzw%kfymjwx%gwtzlmy%ktwym1%ts%ymnx%htsynsjsy1%f%sj|%sfynts1%htshjn{ji%ns%qngjwy~1%fsi%ijinhfyji%yt%ymj%uwtutxnynts%ymfy%fqq%rjs%fwj%hwjfyji%jvzfq3%St|%|j%fwj%jslflji%ns%f%lwjfy%hn{nq%|fw1%yjxynsl%|mjymjw%ymfy%sfynts1%tw%fs~%sfynts%xt%htshjn{ji1%fsi%xt%ijinhfyji1%hfs%qtsl%jsizwj3'

In [113]:
# Deciphering subtracts the same constant.
bytes((x - key) for x in shifted_text)

b'Fourscore and seven years ago our fathers brought forth, on this continent, a new nation, conceived in liberty, and dedicated to the proposition that all men are created equal. Now we are engaged in a great civil war, testing whether that nation, or any nation so conceived, and so dedicated, can long endure.'

### 9.4 Frequency analysis

Brute force worked above, but we do not even need it. In English prose the most common character
by a wide margin is the space. Whatever byte appears most often in the ciphertext is almost
certainly the encrypted space, and the difference between them is the key.

That is a one-shot attack requiring a single subtraction.

In [114]:
# Count byte frequencies in the ciphertext and show the three most common.
from collections import Counter
Counter(shifted_text).most_common(3)

[(37, 52), (106, 32), (115, 27)]

In [115]:
# Compute the key from frequency analysis.
# The most common byte is assumed to be the encrypted space character.
space_guess = Counter(shifted_text).most_common(1)[0][0]
print("Most common ciphertext byte:", space_guess, struct.pack("B", space_guess))
space = ord(' ')
print("ASCII space:", space)
key_guess = space_guess - space
print("Recovered key:", key_guess)

Most common ciphertext byte: 37 b'%'
ASCII space: 32
Recovered key: 5


In [116]:
# Verify the recovered key by deciphering with it.
bytes((x - key_guess) for x in shifted_text)[:60]

b'Fourscore and seven years ago our fathers brought forth, on '

In [122]:
# ROT13 is a Caesar shift of 13 applied to letters only.
# Because 13 is half of 26, encoding and decoding are the same operation.
import codecs
rot = codecs.encode('aAbcdez', 'rot13')
rot

'nNopqrm'

In [123]:
codecs.encode(rot, 'rot13')

'aAbcdez'

---
## 10. Examine Different Character Encodings

A closing look at how much the choice of codec changes the story a byte tells. Each cell below
renders all 256 byte values through a different codec.

Bytes that a codec cannot handle are dropped by the `'ignore'` handler, so gaps in the output show
you which byte values that codec leaves undefined.

In [124]:
# UTF-8: only 0x00-0x7F decode as single bytes; the rest are dropped.
" ".join([struct.pack('B', x).decode('utf-8', 'ignore') for x in range(0xff)])

'\x00 \x01 \x02 \x03 \x04 \x05 \x06 \x07 \x08 \t \n \x0b \x0c \r \x0e \x0f \x10 \x11 \x12 \x13 \x14 \x15 \x16 \x17 \x18 \x19 \x1a \x1b \x1c \x1d \x1e \x1f   ! " # $ % & \' ( ) * + , - . / 0 1 2 3 4 5 6 7 8 9 : ; < = > ? @ A B C D E F G H I J K L M N O P Q R S T U V W X Y Z [ \\ ] ^ _ ` a b c d e f g h i j k l m n o p q r s t u v w x y z { | } ~ \x7f                                                                                                                               '

In [125]:
# Latin-1 (ISO 8859-1): every one of the 256 values maps to a character.
" ".join([struct.pack('B', x).decode('latin-1', 'ignore') for x in range(0xff)])

'\x00 \x01 \x02 \x03 \x04 \x05 \x06 \x07 \x08 \t \n \x0b \x0c \r \x0e \x0f \x10 \x11 \x12 \x13 \x14 \x15 \x16 \x17 \x18 \x19 \x1a \x1b \x1c \x1d \x1e \x1f   ! " # $ % & \' ( ) * + , - . / 0 1 2 3 4 5 6 7 8 9 : ; < = > ? @ A B C D E F G H I J K L M N O P Q R S T U V W X Y Z [ \\ ] ^ _ ` a b c d e f g h i j k l m n o p q r s t u v w x y z { | } ~ \x7f \x80 \x81 \x82 \x83 \x84 \x85 \x86 \x87 \x88 \x89 \x8a \x8b \x8c \x8d \x8e \x8f \x90 \x91 \x92 \x93 \x94 \x95 \x96 \x97 \x98 \x99 \x9a \x9b \x9c \x9d \x9e \x9f \xa0 ¡ ¢ £ ¤ ¥ ¦ § ¨ © ª « ¬ \xad ® ¯ ° ± ² ³ ´ µ ¶ · ¸ ¹ º » ¼ ½ ¾ ¿ À Á Â Ã Ä Å Æ Ç È É Ê Ë Ì Í Î Ï Ð Ñ Ò Ó Ô Õ Ö × Ø Ù Ú Û Ü Ý Þ ß à á â ã ä å æ ç è é ê ë ì í î ï ð ñ ò ó ô õ ö ÷ ø ù ú û ü ý þ'

In [126]:
# Greek (ISO 8859-7): the same upper bytes, an entirely different alphabet.
" ".join([struct.pack('B', x).decode('greek', 'ignore') for x in range(0xff)])

'\x00 \x01 \x02 \x03 \x04 \x05 \x06 \x07 \x08 \t \n \x0b \x0c \r \x0e \x0f \x10 \x11 \x12 \x13 \x14 \x15 \x16 \x17 \x18 \x19 \x1a \x1b \x1c \x1d \x1e \x1f   ! " # $ % & \' ( ) * + , - . / 0 1 2 3 4 5 6 7 8 9 : ; < = > ? @ A B C D E F G H I J K L M N O P Q R S T U V W X Y Z [ \\ ] ^ _ ` a b c d e f g h i j k l m n o p q r s t u v w x y z { | } ~ \x7f \x80 \x81 \x82 \x83 \x84 \x85 \x86 \x87 \x88 \x89 \x8a \x8b \x8c \x8d \x8e \x8f \x90 \x91 \x92 \x93 \x94 \x95 \x96 \x97 \x98 \x99 \x9a \x9b \x9c \x9d \x9e \x9f \xa0 ‘ ’ £ € ₯ ¦ § ¨ © ͺ « ¬ \xad  ― ° ± ² ³ ΄ ΅ Ά · Έ Ή Ί » Ό ½ Ύ Ώ ΐ Α Β Γ Δ Ε Ζ Η Θ Ι Κ Λ Μ Ν Ξ Ο Π Ρ  Σ Τ Υ Φ Χ Ψ Ω Ϊ Ϋ ά έ ή ί ΰ α β γ δ ε ζ η θ ι κ λ μ ν ξ ο π ρ ς σ τ υ φ χ ψ ω ϊ ϋ ό ύ ώ'

---
## Concluding Remarks

* You should now see and appreciate the different ways binary data can be encoded.
* Communications rely heavily on shared codecs and data definitions. The bytes carry no meaning
  by themselves; the standard supplies it.
* **Encoding is not encrypting.** Base64, hex, and ASCII are presentation choices, not
  protections.
* Byte order and signedness are not details. Guess wrong and you will get an answer that looks
  like data and is not.
* When a decoded value is absurd, suspect your interpretation before you suspect the sensor.